# Uji Validitas dan Reliabilitas (VS Code)

Letakkan file `Blank_Quiz (Responses).xlsx` di folder yang sama dengan notebook ini (atau ubah `NAMA_FILE` di Sel 3).

Pustaka yang dibutuhkan: `pandas`, `numpy`, `scipy`, `openpyxl`, `ipykernel`.
Jika belum ada, jalankan di terminal: `pip install pandas numpy scipy openpyxl ipykernel`

## Sel 1: impor pustaka

In [6]:
%pip install pandas numpy scipy openpyxl

Note: you may need to restart the kernel to use updated packages.


In [7]:
import os
import re
import pandas as pd
import numpy as np
from scipy import stats
from IPython.display import display

## Sel 2: tentukan lokasi berkas

Pengganti `files.upload()` pada Colab. Berkas dibaca langsung dari folder kerja.

In [8]:
NAMA_FILE = 'Blank Quiz (Responses).xlsx'   # ubah jika nama/lokasi berbeda

if not os.path.exists(NAMA_FILE):
    raise FileNotFoundError(
        f'File {NAMA_FILE!r} tidak ditemukan di {os.getcwd()}. '
        'Letakkan file di folder yang sama dengan notebook atau isi path lengkap.'
    )

nama_file = NAMA_FILE
print('File terbaca:', nama_file)

File terbaca: Blank Quiz (Responses).xlsx


## Sel 3: baca data dan pisahkan butir

In [9]:
JUMLAH_KOLOM_DEMOGRAFI = 6  # Timestamp, Score, Kelas, Jenis Kelamin, 2 kolom jumlah platform AI

df = pd.read_excel(nama_file)
butir = df.iloc[:, JUMLAH_KOLOM_DEMOGRAFI:]

# Buang kolom yang bukan butir Likert (tanpa format 'Dimensi [pernyataan]'),
# yaitu kolom 'Durasi (jam) penggunaan gadget'
kolom_bukan_butir = [k for k in butir.columns if '[' not in k]
print('Kolom dikeluarkan:', kolom_bukan_butir)
butir = butir.drop(columns=kolom_bukan_butir)

print('Jumlah responden :', len(df))
print('Jumlah butir     :', butir.shape[1])
print('Data kosong      :', int(butir.isna().sum().sum()))

Kolom dikeluarkan: ['Durasi (jam) rata-rata penggunaan gadget perhari, untuk , kebutuhan harian sampai dengan hiburan ']
Jumlah responden : 38
Jumlah butir     : 29
Data kosong      : 0


Keluaran yang diharapkan: 1 kolom dikeluarkan (Durasi), responden 38, butir 29, data kosong 0.

## Sel 4: ubah jawaban Likert menjadi angka

In [10]:
skala = {
    'Sangat Tidak Setuju (STS)': 1,
    'Tidak Setuju (TS)': 2,
    'Netral (N)': 3,
    'Setuju (S)': 4,
    'Sangat Setuju (SS)': 5,
}

num = butir.replace(skala).apply(pd.to_numeric, errors='coerce')

# Harus 0. Jika lebih dari 0, ada jawaban yang tidak terbaca.
print('Sel yang gagal dikonversi:', int(num.isna().sum().sum()))
num.head()

Sel yang gagal dikonversi: 0


,Penggunaan AI pada tugas atau projek kuliah [Saya menggunakan platform AI hampir setiap kali mendapatkan tugas akademik.],Penggunaan AI pada tugas atau projek kuliah [Saya menggunakan AI untuk mencari solusi atas permasalahan atau soal perkuliahan.],Penggunaan AI pada tugas atau projek kuliah [Saya menggunakan AI untuk mencari ide atau inspirasi dalam mengerjakan tugas./projek],Penggunaan AI pada tugas atau projek kuliah [Saya merasa kesulitan memulai mengerjakan tugas tanpa bantuan AI.],Penggunaan AI pada tugas atau projek kuliah [Saya merasa kurang yakin dengan hasil tugas apabila tidak menggunakan AI.],Penggunaan AI pada tugas atau projek kuliah [Saya percaya bahwa AI dapat memberikan informasi yang membantu penyelesaian tugas.],Penggunaan AI pada tugas atau projek kuliah [Saya memeriksa kembali kebenaran informasi yang diberikan AI sebelum menggunakannya.],Aspek Psikologis penggunaan AI pada tugas atau projek kuliah [Saya merasa perlu menggunakan AI setiap kali mendapatkan tugas atau proyek kuliah.],Aspek Psikologis penggunaan AI pada tugas atau projek kuliah [aya merasa kesulitan menyelesaikan tugas kuliah apabila tidak menggunakan bantuan AI.],Aspek Psikologis penggunaan AI pada tugas atau projek kuliah [Saya cenderung mengikuti saran atau jawaban AI tanpa banyak mempertimbangkan alternatif lainnya (Non-AI).],...,Kebiasaan penggunaan teknologi digital [Saya jarang memeriksa kembali hasil dari AI karena sudah percaya terhadap ketepatan jawabannya.],Prokrastinasi akademik. [Saya cenderung menunggu hingga mendekati deadline untuk mulai mengerjakan tugas.],Prokrastinasi akademik. [Saya sering menunda pengerjaan tugas karena merasa dapat dikerjakan dengan mudah dan cepat dengan menggunakan AI],Prokrastinasi akademik. [Saya sering menyelesaikan tugas kuliah dalam waktu yang sangat dekat dengan batas pengumpulan.],Prokrastinasi akademik. [Saya sering mengerjakan beberapa tugas sekaligus karena sebelumnya menunda pengerjaannya.],Rsspon emosional [Saya merasa tertekan ketika harus menyelesaikan tugas dalam waktu singkat akibat menunda pengerjaannya.],Rsspon emosional [Saya merasa panik ketika beberapa tugas yang saya tunda memiliki batas waktu pengumpulan yang berdekatan.],Rsspon emosional [Saya merasa kurang puas terhadap hasil tugas yang dikerjakan karena terburu-buru mendekati deadline.],Rsspon emosional [Saya merasa lega setelah berhasil mengumpulkan tugas meskipun pengerjaannya dilakukan AI saat mendekati deadline.],Rsspon emosional [Saya merasa menyesal karena hasil tugas mungkin bisa lebih baik apabila dikerjakan lebih awal.]
0,3,4,4,4,3,3,4,3,4,3,...,3,3,3,3,4,4,3,4,4,3
1,4,4,4,2,3,4,5,3,2,1,...,2,3,3,3,4,4,4,4,3,4
2,3,4,5,3,4,4,5,4,4,3,...,2,3,3,3,3,4,4,3,4,4
3,5,5,5,5,4,5,5,5,4,5,...,5,5,4,4,5,4,5,4,5,5
4,5,4,4,3,4,4,4,3,4,4,...,2,4,4,3,4,3,4,5,4,5


## Sel 5: kelompokkan butir per dimensi

In [11]:
dimensi = {}
for kolom in butir.columns:
    nama = re.match(r'^(.*?) \[', kolom).group(1).strip()
    dimensi.setdefault(nama, []).append(kolom)

for nama, daftar in dimensi.items():
    print(f'{nama}: {len(daftar)} butir')

Penggunaan AI pada tugas atau projek kuliah: 7 butir
Aspek Psikologis penggunaan AI pada tugas atau projek kuliah: 7 butir
Kebiasaan penggunaan teknologi digital: 6 butir
Prokrastinasi akademik.: 4 butir
Rsspon emosional: 5 butir


Keluaran yang diharapkan: 5 dimensi dengan jumlah butir 7, 7, 6, 4, 5.

## Sel 6: uji validitas

In [12]:
n = len(num)
t = stats.t.ppf(0.975, n - 2)
r_tabel = t / np.sqrt((n - 2) + t**2)
print(f'n = {n} | r tabel (α=0,05; df={n-2}) = {r_tabel:.3f}')

hasil_validitas = []
for nama, daftar in dimensi.items():
    x = num[daftar]
    total = x.sum(axis=1)
    for i, kolom in enumerate(daftar, 1):
        r, p = stats.pearsonr(x[kolom], total)
        r_koreksi, _ = stats.pearsonr(x[kolom], total - x[kolom])
        hasil_validitas.append({
            'Dimensi': nama,
            'Butir': f'B{i}',
            'r hitung': round(r, 3),
            'Corrected r': round(r_koreksi, 3),
            'p-value': round(p, 4),
            'Keterangan': 'Valid' if r > r_tabel else 'Tidak valid',
        })

tabel_validitas = pd.DataFrame(hasil_validitas)
tabel_validitas

n = 38 | r tabel (α=0,05; df=36) = 0.320


,Dimensi,Butir,r hitung,Corrected r,p-value,Keterangan
0,Penggunaan AI pada tugas atau projek kuliah,B1,0.795,0.705,0.0000,Valid
1,Penggunaan AI pada tugas atau projek kuliah,B2,0.744,0.641,0.0000,Valid
2,Penggunaan AI pada tugas atau projek kuliah,B3,0.797,0.713,0.0000,Valid
3,Penggunaan AI pada tugas atau projek kuliah,B4,0.464,0.330,0.0033,Valid
4,Penggunaan AI pada tugas atau projek kuliah,B5,0.772,0.662,0.0000,Valid
5,Penggunaan AI pada tugas atau projek kuliah,B6,0.838,0.756,0.0000,Valid
6,Penggunaan AI pada tugas atau projek kuliah,B7,0.757,0.644,0.0000,Valid
7,Aspek Psikologis penggunaan AI pada tugas atau...,B1,0.861,0.800,0.0000,Valid
8,Aspek Psikologis penggunaan AI pada tugas atau...,B2,0.793,0.715,0.0000,Valid
9,Aspek Psikologis penggunaan AI pada tugas atau...,B3,0.763,0.662,0.0000,Valid


Keluaran yang diharapkan: r tabel = 0,320, dan 29 butir berstatus Valid.

## Sel 7: uji reliabilitas

In [13]:
def cronbach_alpha(x):
    k = x.shape[1]
    return k / (k - 1) * (1 - x.var(ddof=1).sum() / x.sum(axis=1).var(ddof=1))

def kategori(a):
    return ('Sangat baik' if a >= 0.90 else 'Baik' if a >= 0.80
            else 'Reliabel' if a >= 0.70 else 'Tidak reliabel')

hasil_reliabilitas = []
alpha_dihapus = []
for nama, daftar in dimensi.items():
    x = num[daftar]
    a = cronbach_alpha(x)
    hasil_reliabilitas.append({
        'Dimensi': nama,
        'Jumlah butir': len(daftar),
        "Cronbach's alpha": round(a, 3),
        'Kategori': kategori(a),
    })
    for i, kolom in enumerate(daftar, 1):
        if len(daftar) > 2:
            alpha_dihapus.append({
                'Dimensi': nama,
                'Butir': f'B{i}',
                'Alpha jika dihapus': round(cronbach_alpha(x.drop(columns=kolom)), 3),
            })

tabel_reliabilitas = pd.DataFrame(hasil_reliabilitas)
tabel_alpha_dihapus = pd.DataFrame(alpha_dihapus)
display(tabel_reliabilitas)
display(tabel_alpha_dihapus)

,Dimensi,Jumlah butir,Cronbach's alpha,Kategori
0,Penggunaan AI pada tugas atau projek kuliah,7,0.865,Baik
1,Aspek Psikologis penggunaan AI pada tugas atau...,7,0.904,Sangat baik
2,Kebiasaan penggunaan teknologi digital,6,0.777,Reliabel
3,Prokrastinasi akademik.,4,0.851,Baik
4,Rsspon emosional,5,0.873,Baik


,Dimensi,Butir,Alpha jika dihapus
0,Penggunaan AI pada tugas atau projek kuliah,B1,0.837
1,Penggunaan AI pada tugas atau projek kuliah,B2,0.846
2,Penggunaan AI pada tugas atau projek kuliah,B3,0.836
3,Penggunaan AI pada tugas atau projek kuliah,B4,0.881
4,Penggunaan AI pada tugas atau projek kuliah,B5,0.843
5,Penggunaan AI pada tugas atau projek kuliah,B6,0.829
6,Penggunaan AI pada tugas atau projek kuliah,B7,0.846
7,Aspek Psikologis penggunaan AI pada tugas atau...,B1,0.879
8,Aspek Psikologis penggunaan AI pada tugas atau...,B2,0.889
9,Aspek Psikologis penggunaan AI pada tugas atau...,B3,0.896


Keluaran yang diharapkan: α per dimensi 0,865 / 0,904 / 0,777 / 0,851 / 0,873.

## Sel 8: gabungkan dan simpan hasil

Pengganti `files.download()`: hasil disimpan sebagai file Excel di folder kerja.

In [14]:
gabung = tabel_validitas.merge(tabel_alpha_dihapus, on=['Dimensi', 'Butir'], how='left')

FILE_HASIL = 'hasil_uji_validitas_reliabilitas.xlsx'
with pd.ExcelWriter(FILE_HASIL) as w:
    gabung.to_excel(w, sheet_name='Validitas', index=False)
    tabel_reliabilitas.to_excel(w, sheet_name='Reliabilitas', index=False)

print('Hasil disimpan di:', os.path.abspath(FILE_HASIL))

Hasil disimpan di: d:\Semester5\ProbStat\hasil_uji_validitas_reliabilitas.xlsx


## Sel 9: Latihan 2 (perbandingan kriteria)

In [15]:
print('Valid berdasarkan r hitung        :', (tabel_validitas['r hitung'] > r_tabel).sum())
print('Valid berdasarkan corrected r≥0,30:', (tabel_validitas['Corrected r'] >= 0.30).sum())

Valid berdasarkan r hitung        : 29
Valid berdasarkan corrected r≥0,30: 29


Keluaran yang diharapkan: 29 dan 29.

## Sel 10: Latihan 3 (buang satu butir dan hitung ulang)

In [16]:
nama_dim = 'Penggunaan AI pada tugas atau projek kuliah'
semua = dimensi[nama_dim]
sisa = [k for i, k in enumerate(semua) if i != 3]   # buang B4

print('Alpha sebelum:', round(cronbach_alpha(num[semua]), 3))
print('Alpha sesudah:', round(cronbach_alpha(num[sisa]), 3))
print(f'r tabel = {r_tabel:.3f}\n')

x = num[sisa]
total = x.sum(axis=1)
for i, k in enumerate(sisa, 1):
    r, _ = stats.pearsonr(x[k], total)
    rk, _ = stats.pearsonr(x[k], total - x[k])
    print(f'B{i}: r hitung={r:.3f} | corrected r={rk:.3f}')

Alpha sebelum: 0.865
Alpha sesudah: 0.881
r tabel = 0.320

B1: r hitung=0.796 | corrected r=0.699
B2: r hitung=0.753 | corrected r=0.644
B3: r hitung=0.820 | corrected r=0.738
B4: r hitung=0.738 | corrected r=0.606
B5: r hitung=0.865 | corrected r=0.789
B6: r hitung=0.785 | corrected r=0.673


Keluaran yang diharapkan: alpha sebelum 0,865, sesudah 0,881. Keenam butir sisanya tetap valid.

## Sel 11: Latihan 4 (pengaruh ukuran sampel)

In [17]:
sampel = num.sample(20, random_state=1)
n2 = len(sampel)
t2 = stats.t.ppf(0.975, n2 - 2)
r_tabel2 = t2 / np.sqrt((n2 - 2) + t2**2)

jumlah_valid = 0
for nama, daftar in dimensi.items():
    x = sampel[daftar]
    total = x.sum(axis=1)
    for k in daftar:
        r, _ = stats.pearsonr(x[k], total)
        jumlah_valid += int(r > r_tabel2)

print(f'n = {n2} | r tabel = {r_tabel2:.3f} | butir valid = {jumlah_valid} dari {num.shape[1]}')

n = 20 | r tabel = 0.444 | butir valid = 29 dari 29


Keluaran yang diharapkan: r tabel = 0,444, butir valid = 29 dari 29.

In [18]:
# Statistika deskriptif per dimensi: varians, skewness, kurtosis
hasil_deskriptif = []
for nama, daftar in dimensi.items():
    skor = num[daftar].mean(axis=1)      # skor rata-rata butir per responden
    # Jika ingin skor total dimensi, ganti baris di atas dengan:
    # skor = num[daftar].sum(axis=1)
    hasil_deskriptif.append({
        'Dimensi': nama,
        'Jumlah butir': len(daftar),
        'Varians': round(skor.var(ddof=1), 3),      # varians sampel (n-1)
        'Skewness': round(skor.skew(), 3),          # koreksi bias sampel
        'Kurtosis': round(skor.kurt(), 3),          # excess kurtosis (normal = 0)
    })

tabel_deskriptif = pd.DataFrame(hasil_deskriptif)
tabel_deskriptif

,Dimensi,Jumlah butir,Varians,Skewness,Kurtosis
0,Penggunaan AI pada tugas atau projek kuliah,7,0.458,-1.186,3.586
1,Aspek Psikologis penggunaan AI pada tugas atau...,7,0.546,-0.101,1.784
2,Kebiasaan penggunaan teknologi digital,6,0.421,0.688,1.356
3,Prokrastinasi akademik.,4,0.614,-0.260,0.839
4,Rsspon emosional,5,0.627,-1.460,2.513
